# Диаризация аудио с pyannote.audio

### Установка зависимостей

При работе в google collab -> После первой установки выбрать **Среда выполнения -> Перезапустить сеанс**, затем продолжить с ячейки инициализации ниже, не запуская установку повторно.

In [ ]:
%pip install -q "pyannote.audio==4.0.4"

### Инициализация девайса и библиотек

При работе в google collab. После перезапуска начать отсюда.

In [ ]:
import json
import subprocess
from getpass import getpass
from pathlib import Path

import pandas as pd
import torch
from google.colab import files, userdata
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU доступен: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найден. Будет использован CPU.")

### Настройка токена Hugging Face

Создать аккаунт Hugging Face, принять условия модели [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1), создать токен с правом чтения и добавить его в секреты Colab под именем HF_TOKEN.

In [ ]:
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "Токен Hugging Face отсутствует. "
    )
print("Токен получен.")

### Загрузка аудиофайла

Выбрать один аудиофайл или медиаконтейнер с аудиодорожкой из числа поддерживаемых FFmpeg.

In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("Файл не загружен.")
if len(uploaded) != 1:
    raise ValueError("Загружено несколько файлов, вместо одного.")

audio_name = next(iter(uploaded))
audio_path = Path(audio_name)
allowed_extensions = {
    ".3g2", ".3gp", ".aac", ".ac3", ".aif", ".aifc", ".aiff",
    ".amr", ".ape", ".au", ".avi", ".awb", ".caf", ".dts",
    ".eac3", ".flac", ".flv", ".gsm", ".m2ts", ".m4a", ".m4b",
    ".mka", ".mkv", ".mov", ".mp2", ".mp3", ".mp4", ".mpc",
    ".mpeg", ".mpg", ".mts", ".oga", ".ogg", ".opus", ".ra",
    ".rm", ".snd", ".spx", ".tak", ".ts", ".tta", ".wav",
    ".wave", ".webm", ".wma", ".wv",
}
if audio_path.suffix.lower() not in allowed_extensions:
    raise ValueError(
        f"Неподдерживаемый формат {audio_path.suffix or 'без расширения'}. "
        "Выберите аудиофайл или медиаконтейнер с поддерживаемой аудиодорожкой."
    )
if not audio_path.is_file() or audio_path.stat().st_size == 0:
    raise ValueError("Загруженный файл отсутствует или пуст.")
print(f"Загружен файл: {audio_name} ({audio_path.stat().st_size / 1024 / 1024:.2f} МБ)")

### Подготовка аудио

In [ ]:
prepared_path = Path("/tmp/prepared_audio.wav")
command = [
    "ffmpeg", "-v", "error", "-y", "-i", str(audio_path),
    "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(prepared_path),
]
try:
    conversion = subprocess.run(command, capture_output=True, text=True, check=False)
except FileNotFoundError as exc:
    raise RuntimeError("FFmpeg not found.") from exc

if conversion.returncode != 0 or not prepared_path.is_file() or prepared_path.stat().st_size <= 44:
    details = (conversion.stderr or "FFmpeg error").strip()[-1000:]
    raise RuntimeError(
        "Не удалось прочитать или преобразовать аудио."
        f"Сообщение FFmpeg: {details}"
    )
print(f"Аудио подготовлено: {prepared_path} (mono, 16 кГц, WAV)")

### Загрузка модели

In [ ]:
from pyannote.audio import Pipeline

MODEL_ID = "pyannote/speaker-diarization-community-1"
try:
    pipeline = Pipeline.from_pretrained(MODEL_ID, token=HF_TOKEN)
    if pipeline is None:
        raise RuntimeError("Pipeline.from_pretrained returned None.")
    pipeline.to(DEVICE)
except Exception as exc:
    message = str(exc)
    raise RuntimeError(f"Не удалось загрузить модель: {message}") from exc
print(f"Модель загружена на {DEVICE}.")

### Выполнение диаризации

При известности числа участников, задать значение NUM_SPEAKERS целым положительным числом. Значение None включает автоматическое определение.

In [ ]:
NUM_SPEAKERS = None

if NUM_SPEAKERS is not None and (isinstance(NUM_SPEAKERS, bool) or not isinstance(NUM_SPEAKERS, int) or NUM_SPEAKERS < 1):
    raise ValueError("NUM_SPEAKERS должен быть положительным целым числом или None.")

inference_options = {} if NUM_SPEAKERS is None else {"num_speakers": NUM_SPEAKERS}
try:
    output = pipeline(str(prepared_path), **inference_options)
except (torch.cuda.OutOfMemoryError, MemoryError) as exc:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise RuntimeError(
        "Во время диаризации закончилась оперативная или видеопамять. "
    ) from exc
except Exception as exc:
    raise RuntimeError(
        f"Диаризация завершилась ошибкой: {exc}."
    ) from exc

segments = [
    {"start": round(float(turn.start), 3), "end": round(float(turn.end), 3), "speaker": str(speaker)}
    for turn, speaker in output.speaker_diarization
]
if not segments:
    raise RuntimeError(
        "В записи не обнаружена речь."
    )
print(f"Диаризация завершена. Найдено сегментов: {len(segments)}")

### Просмотр и скачивание временной разметки

Сегменты выводятся в таблице в минутах, полученная временная разметка сохраняется в json(в секундах).

In [ ]:
def format_minutes(seconds):
    minutes, remaining_seconds = divmod(float(seconds), 60)
    return f"{int(minutes):02d}:{remaining_seconds:06.3f}"

table = pd.DataFrame(
    [
        (format_minutes(item["start"]), format_minutes(item["end"]), item["speaker"])
        for item in segments
    ],
    columns=["Начало", "Окончание", "Говорящий"],
)
with pd.option_context("display.max_rows", None):
    display(table)

result = {"audio_file": audio_name, "segments": segments}
result_path = Path(audio_name).with_suffix(".json")
with result_path.open("w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)

print(f"Результат сохранён: {result_path.resolve()}")
files.download(str(result_path))